# 🦺 ASSIS — PPE Detection Model Training
**Phase 1 MVP**

Trains a YOLOv8 model to detect high-visibility vest compliance for airport ramp operations.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU` (required!)
2. Free Roboflow account ready (roboflow.com)
3. Total runtime: ~1–3 hours

**Workflow:** Install → Dataset → Train → Evaluate → Test → Download weights

## Step 1 — Verify GPU & Install

In [ ]:
# You should see a Tesla T4 or similar
!nvidia-smi

In [ ]:
%pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## Step 2 — Get the Dataset from Roboflow

1. Go to [Roboflow Universe](https://universe.roboflow.com), search **"PPE vest"** or **"safety vest"**
2. Pick a dataset with 1,000+ images and vest/no-vest style classes
3. **Download Dataset → YOLOv8 → Get Snippet**
4. Replace the placeholders below with YOUR api key, workspace, project, version

> 💡 API key: Roboflow → Settings → API Keys

In [ ]:
from roboflow import Roboflow

# ⬇️ REPLACE with your values from the Roboflow snippet
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")
version = project.version(1)
dataset = version.download("yolov8")

print("Dataset downloaded to:", dataset.location)

In [ ]:
import yaml, os

data_yaml = os.path.join(dataset.location, "data.yaml")
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

print("Classes:", cfg["names"])
print("Train:", cfg.get("train"))
print("Val:", cfg.get("val"))

## Step 3 — Train

Fine-tunes **YOLOv8n** (nano). `epochs=50` is a solid start; early-stops automatically if accuracy plateaus. Watch **mAP50** climb — that's your headline accuracy.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    plots=True,
    project="runs",
    name="assis_ppe",
)

## Step 4 — Evaluate

**MVP target: mAP50 ≥ 0.85.** Below target? Add harder training images (low light, occlusion, distance) and retrain.

In [ ]:
metrics = model.val()

print(f"mAP50:    {metrics.box.map50:.3f}   <- headline accuracy")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
from IPython.display import Image as IPImage
IPImage(filename="runs/assis_ppe/results.png", width=900)

## Step 5 — Test on a Sample Image

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick a test image

test_image = list(uploaded.keys())[0]
results = model(test_image, conf=0.4)
results[0].show()

In [ ]:
r = results[0]
names = r.names
counts = {}
for box in r.boxes:
    label = names[int(box.cls[0])]
    counts[label] = counts.get(label, 0) + 1

total = sum(counts.values())
compliant = counts.get("vest", 0)
violations = total - compliant

print(f"Personnel detected: {total}")
print(f"Wearing vest:       {compliant}")
print(f"Violations:         {violations}")
if total:
    print(f"Compliance rate:    {100*compliant/total:.1f}%")

## Step 6 — Download Trained Weights

`best.pt` is your trained model — put it in the repo's `models/` folder to power the Streamlit demo.

In [ ]:
from google.colab import files
files.download("runs/assis_ppe/weights/best.pt")

## ✅ Next Steps

1. `best.pt` → repo `models/` folder
2. `streamlit run app/streamlit_app.py`
3. Record metrics (mAP50, precision, recall) → white paper Results section
4. Screenshot the demo → portfolio/ACRP evidence
5. Commit to GitHub — timestamped progress
6. Log the session in `docs/RESEARCH_LOG.md`

---
*ASSIS Research Project · Phase 1 · FAA Part 139 SMS & TSA Part 1542 aligned*